In [ ]:
# coding: utf-8
import sys
sys.path.append('..')
from common.np import *  # import numpy as np
from common.layers import Embedding, SigmoidWithLoss
import collections


class EmbeddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)  # Embeddingクラスをself.embedとして呼び出せるようにする
        self.params = self.embed.params  # self.embed.paramsにある重みを、self.paramsとして呼び出す
        self.grads = self.embed.grads  # self.embed.gradsにある勾配を、self.gradsとして呼び出す
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)  # idx番目の重みを抜き出して、target_Wとして呼び出す
        out = np.sum(target_W * h, axis=1)  # target_Wと周りの単語を混ぜたベクトル（h）を掛けてスコアを出す

        self.cache = (h, target_W)  # 後のbackwardで使うため、スコアの材料をself.cacheで呼び出せるようにしておく
        return out

    def backward(self, dout):
        h, target_W = self.cache  # スコアの材料を取り出す
        dout = dout.reshape(dout.shape[0], 1)  # doutを、ベクトルとの掛け算ができる形に変形する

        dtarget_W = dout * h  # doutにhをかけて勾配を計算する
        self.embed.backward(dtarget_W)  # dtarget_Wの勾配をEmbeddingレイヤに渡す
        dh = dout * target_W # ズレに単語の重みを掛ける
        return dh


class UnigramSampler:
    def __init__(self, corpus, power, sample_size):
        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        self.word_p = np.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        self.word_p = np.power(self.word_p, power)
        self.word_p /= np.sum(self.word_p)

    def get_negative_sample(self, target):
        batch_size = target.shape[0]

        if not GPU:
            negative_sample = np.zeros((batch_size, self.sample_size), dtype=np.int32)

            for i in range(batch_size):
                p = self.word_p.copy()
                target_idx = target[i]
                p[target_idx] = 0
                p /= p.sum()
                negative_sample[i, :] = np.random.choice(self.vocab_size, size=self.sample_size, replace=False, p=p)
        else:
            # GPU(cupy）で計算するときは、速度を優先
            # 負例にターゲットが含まれるケースがある
            negative_sample = np.random.choice(self.vocab_size, size=(batch_size, self.sample_size),
                                               replace=True, p=self.word_p)

        return negative_sample


class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)  # corpusから5つハズレを選ぶ
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)] # 判定機を6つ分用意
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)] # 計算機を6つ分用意

        self.params, self.grads = [], []    # 重みと勾配を1つのリストにまとめる
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)  # ハズレ単語を5つ選ぶ指示を出す

        # 正例のフォワード
        score = self.embed_dot_layers[0].forward(h, target)  # 0番目に、targetとスコアを出させる
        correct_label = np.ones(batch_size, dtype=np.int32)  # 正解を1に設定する
        loss = self.loss_layers[0].forward(score, correct_label)  # 0番目がスコア1に近いかを判定、最初のズレを出す

        # 負例のフォワード
        negative_label = np.zeros(batch_size, dtype=np.int32)
        for i in range(self.sample_size):
            negative_target = negative_sample[:, i]
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)
            loss += self.loss_layers[1 + i].forward(score, negative_label)

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh

In [ ]:
class Embedding:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None

    def forward(self, idx):
        W, = self.params   # W = self.params[0]と同じ意味
        self.idx = idx
        out = W[idx]   # Wからidx番目の重みを取り出す
        return out

    def backward(self, dout):
        dW, = self.grads
        dW[...] = 0
        if GPU:
            np.scatter_add(dW, self.idx, dout)
        else:
            np.add.at(dW, self.idx, dout)
        return None